# gridkit helper v1
* gridkit installed in:
* /home/isatkaus/gridkit

#### instructions
* /home/isatkaus/gridkit/uq-usecase/kestrel_install.md

#### context - needs revisiting
* /gridkit_docs 
* /home/isatkaus/.github/prompts/gridkit_docs.prompt.md
* has links to all gridkit doc .md files
* file is made every time we rebuild the gridkit mkdocs


In [ ]:
lis = [
    "sdcs",
    "sdf",
    "sdf",
]

In [ ]:
from IPython.display import Image

# import cufflinks as cl
import pandas as pd
import numpy as np
import os
import sys
import shutil
import re
import math
import json5
import datetime

import re
import textwrap

# import reV
# import PySAM


# import matplotlib.pyplot as plt
import glob
import os
import h5py
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp

# show multiple cell outputs
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

#### default values
n = pd.get_option("display.max_rows")
m = pd.get_option("display.max_columns")
k = pd.get_option("display.max_colwidth")

pd.set_option("display.max_rows", 10)  # 60 default
pd.set_option("display.max_columns", 100)  # 20 default
pd.set_option("display.max_colwidth", 60)  # 50 default


### use to use df.plot instead of df.iplot() - no need for cufflinks
pd.options.plotting.backend = "plotly"

import subprocess

# === GridKit paths ===
GRIDKIT_REPO_ROOT = os.path.expanduser("~/gridkit")
GRIDKIT_BUILD_DIR = os.path.join(GRIDKIT_REPO_ROOT, "build")
GRIDKIT_PY_UTILS = os.path.join(GRIDKIT_REPO_ROOT, "uq-usecase/py-utils")
if GRIDKIT_PY_UTILS not in sys.path:
    sys.path.insert(0, GRIDKIT_PY_UTILS)
print(f"GRIDKIT_REPO_ROOT: {GRIDKIT_REPO_ROOT}")
print(f"GRIDKIT_BUILD_DIR: {GRIDKIT_BUILD_DIR}")
print(f"GRIDKIT_PY_UTILS:  {GRIDKIT_PY_UTILS}")

In [ ]:
from IPython.display import Markdown, display


def printmd(string):
    display(Markdown(string))


printmd("**This text is bold and generated from a code cell.**")


def get_ipynb_name():
    try:
        ipynb_name = os.path.basename(globals()["__vsc_ipynb_file__"])
        # print(ipynb_name)
    except KeyError:
        ipynb_name = "unknown"
    return ipynb_name


ipynb_name = get_ipynb_name()
print(f"\nthis notebook name is: {ipynb_name}\n")


### helper to bold a row in a dataframe - useful for highlighting a specific row in a table
def bold_selected_row(df, row_idx):
    """Bold the row at the specified index in a DataFrame."""

    def highlight_row(row):
        return ["font-weight: bold" if row.name == row_idx else "" for _ in row]

    return df.style.apply(highlight_row, axis=1)


#### example usage:
df = pd.DataFrame({"A": [1, 2, 3], "B": [4, 5, 6], "C": [7, 8, 9]})
print("bold selected row")
bold_selected_row(df, 1)  # This will bold the second row (index 1)

# contents

[Jump to: basic plots](#basic-plots)

[Jump to: end](#end)

## gridkit runner 
* setup DynamicsSimulation 

In [ ]:
build_dir = GRIDKIT_BUILD_DIR
print(f"gridkit build_dir: {build_dir}")

# runner = os.path.join(build_dir, "application/PhasorDynamics/PDSim")
runner = os.path.join(build_dir, "application/PhasorDynamics/DynamicSimulation")
print(
    f"Phasor Dynamics runner, DynamicSimulation, that takes in .solver.json as input:\n {runner}"
)

# Add DynamicSimulation to PATH so it can be called without the full path
os.environ["PATH"] = os.path.dirname(runner) + os.pathsep + os.environ["PATH"]
print(
    f"\nDynamicSimulation added to PATH. You can now run: DynamicSimulation <solver.json>"
)

In [ ]:
!which DynamicSimulation

# simple run

Workflow:
1. Uncomment the `rel_dir` for the case you want to run
2. Set `test_run_dir` (case files are copied here and `DynamicSimulation` runs here); set `output_dir` separately if you want outputs elsewhere (defaults to same as `test_run_dir`)
3. Run the next two cells: first copies files and sets filenames, second executes the runner


In [ ]:
# === 1. select case (uncomment one) ===
# EXAMPLE_DIR = os.path.join(build_dir, "examples/PhasorDynamics/Tiny/ThreeBus/Basic/")
# EXAMPLE_DIR = os.path.join(build_dir, "examples/PhasorDynamics/Tiny/ThreeBus/ZipLoad/")
# EXAMPLE_DIR = os.path.join(build_dir, "examples/PhasorDynamics/Medium/Hawaii/")
EXAMPLE_DIR = os.path.join(build_dir, "examples/PhasorDynamics/Large/Illinois/")

case_name = EXAMPLE_DIR.rstrip("/").split("/")[-1]

# === 2. set run and output dirs ===
test_run_dir = f"./test-runs/{case_name}-v1/"
output_dir = test_run_dir  # change if you want outputs elsewhere

# === solver/case filenames per case ===
_fn_map = {
    "Basic": ("ThreeBusBasic.solver.json", "ThreeBusBasic.case.json"),
    "ZipLoad": ("ThreeBusZipLoad.solver.json", "ThreeBusZipLoad.case.json"),
    "Hawaii": ("hawaii.solver.json", "hawaii.json"),
    "Illinois": ("illinois.solver.json", "illinois.json"),
}
solver_fn, case_fn = _fn_map[case_name]

# === 3. solver overrides (applied to copied solver.json before running) ===
# None = use base case values unchanged
# tmax: simulation end time (s)
# events: list of {index, ...fields} -- patch events by 0-based index
# SOLVER_OVERRIDES = None
SOLVER_OVERRIDES = {
    "tmax": 10.0,
    "events": [
        {"index": 0, "time": 1.0},  # fault_on
        {"index": 1, "time": 1.1},  # fault_off
    ],
}

print(f"case:             {case_name}")
print(f"EXAMPLE_DIR:      {EXAMPLE_DIR}")
print(f"test_run_dir:     {test_run_dir}")
print(f"output_dir:       {output_dir}")
print(f"solver_fn:        {solver_fn}")
print(f"case_fn:          {case_fn}")
print(f"SOLVER_OVERRIDES: {SOLVER_OVERRIDES}")

In [ ]:
# === 4. copy case files to test_run_dir, apply solver overrides, and run ===
import json

os.makedirs(test_run_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

copied = []
for fn in os.listdir(EXAMPLE_DIR):
    src = os.path.join(EXAMPLE_DIR, fn)
    if os.path.isfile(src):
        shutil.copy(src, os.path.join(test_run_dir, fn))
        copied.append(fn)
print(f"Copied {len(copied)} files to {os.path.abspath(test_run_dir)}:")
for fn in sorted(copied):
    print(f"  {fn}")

# apply solver overrides to the copied solver.json
solver_path = os.path.join(test_run_dir, solver_fn)
if SOLVER_OVERRIDES:
    with open(solver_path) as f:
        solver_data = json.load(f)
    if "tmax" in SOLVER_OVERRIDES:
        solver_data["tmax"] = SOLVER_OVERRIDES["tmax"]
    if "events" in SOLVER_OVERRIDES:
        for patch in SOLVER_OVERRIDES["events"]:
            idx = patch["index"]
            for k, v in patch.items():
                if k != "index":
                    solver_data["events"][idx][k] = v
    with open(solver_path, "w") as f:
        json.dump(solver_data, f, indent=4)
    print(f"\nApplied SOLVER_OVERRIDES to {solver_path}:")
    print(json.dumps(solver_data, indent=2))
else:
    print(f"\nSOLVER_OVERRIDES = None, using base solver.json as-is")
    print(open(solver_path).read())

result = subprocess.run(
    ["DynamicSimulation", solver_fn],
    cwd=test_run_dir,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
else:
    outputs = glob.glob(os.path.join(output_dir, "*.csv"))
    print(f"Output files: {outputs}")

# edit case files (optional)
* hawaii generator dispatch

In [ ]:
# # Import dispatch helpers from gridkit_utils
# import importlib
# import gridkit_utils

# importlib.reload(gridkit_utils)
# from gridkit_utils import (
#     get_case_path_for_editing,
#     read_genrou_dispatch,
#     plot_genrou_dispatch,
#     patch_genrou_dispatch,
# )

In [ ]:
# # Read and plot current Genrou dispatch defaults (from copied run dir case file)
# case_path_for_edit = get_case_path_for_editing(test_run_dir, EXAMPLE_DIR, case_fn)
# case_name = case_fn.split(".")[0]
# print(f"Case path for dispatch edits: {case_path_for_edit}")

# dispatch_df = read_genrou_dispatch(case_path_for_edit)
# print(f"Genrou rows: {len(dispatch_df)}")
# dispatch_df

# plot_genrou_dispatch(dispatch_df, case_name=case_name)

In [ ]:
# ### example patch: overwrite copied case file in test_run_dir
# ### edit this list for your aleatoric dispatch experiment
# dispatch_updates = [
#     {"id": "2_1", "p0": 0.030, "q0": 0.010},
#     {"id": "23_1", "p0": 0.220, "q0": 0.070},
#     {"id": "35_1", "p0": 0.250, "q0": 0.080},
# ]

# changes_df = patch_genrou_dispatch(
#     case_json_path=case_path_for_edit,
#     updates=dispatch_updates,
#     output_case_path=None,  # overwrite copied case file in run dir
# )
# changes_df

# # quick verify after patch
# dispatch_df_after = read_genrou_dispatch(case_path_for_edit)
# dispatch_df_after[dispatch_df_after["gen_id"].isin([u["id"] for u in dispatch_updates])]

In [ ]:
### run from test_run_dir (files were copied there and can be modified before this cell)
result = subprocess.run(
    ["DynamicSimulation", solver_fn],
    cwd=test_run_dir,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
else:
    import glob

    outputs = glob.glob(os.path.join(test_run_dir, "*.csv"))
    print(f"Output files: {outputs}")

In [ ]:
# # ### run via ! (output file is in example_dir)
# #!cd {example_dir} && PDSim ThreeBusBasic.solver.json
# !cd {example_dir} && DynamicSimulation {solver_fn}
# ### this has no solver.json
# #!cd {example_dir} && ./ThreeBusZipLoadJson

In [ ]:
# ################# run via subprocess
# result = subprocess.run(
#     # [runner, "ThreeBusBasic.solver.json"],
#     [runner, "ThreeBusBasic.solver.json"],
#     cwd=example_dir,
#     capture_output=True,
#     text=True,
# )
# print(result.stdout)
# if result.returncode != 0:
#     print("STDERR:", result.stderr)

## output
* DynamicSimulation should output mon.csv
* defined by "monitors" in ThreeBusBasic.case.json

#### mon.csv column naming:
* `Bus_<bus-name>_<quantity>` - `Bus_ALOHA138_Vr`
* `<MachineModel>_<bus-number>_<gen-id>_<state-or-output>` - `Genrou_34_5_delta`

In [ ]:
# Bus_<bus-name>_<quantity>  - Bus_ALOHA138_Vr
# <MachineModel>_<bus-number>_<gen-id>_<state-or-output>  - Genrou_34_5_delta

In [ ]:
# !ls -l {example_dir}

In [ ]:
results_fn = os.path.join(test_run_dir, "mon.csv")
# mon_df = pd.read_csv(os.path.join(example_dir, results_fn),index_col=0)
### cwd
mon_df = pd.read_csv(results_fn, index_col=0)
mon_df

In [ ]:
## plot all df columns vs time
fig = px.line(mon_df, x=mon_df.index, y=mon_df.columns)
fig.update_layout(
    title=f"{results_fn} output",
    xaxis_title="Time (s)",
    yaxis_title="Value",
    legend_title="Variables",
)

In [65]:
# Group and plot mon.csv by (element, variable) using MONITORABLE_VARS_BY_ELEMENT
from gridkit_utils import MONITORABLE_VARS_BY_ELEMENT
from collections import defaultdict

grouped_cols = defaultdict(list)
unmatched = []

for col in mon_df.columns:
    parts = col.split("_")
    if len(parts) < 3:
        unmatched.append(col)
        continue

    if parts[0] == "Bus":
        # Bus_<bus-name>_<quantity>
        element = "Bus"
        var = parts[-1]
    else:
        # <MachineModel>_<bus-number>_<gen-id>_<state-or-output>
        element = parts[0]
        var = parts[-1]

    if (
        element in MONITORABLE_VARS_BY_ELEMENT
        and var in MONITORABLE_VARS_BY_ELEMENT[element]
    ):
        grouped_cols[(element, var)].append(col)
    else:
        unmatched.append(col)

# one plot per (element, var), e.g. Bus-Vi, Bus-Vr, Genrou-delta, Genrou-omega
for element, var_list in MONITORABLE_VARS_BY_ELEMENT.items():
    for var in var_list:
        cols = grouped_cols.get((element, var), [])
        if not cols:
            continue

        fig = px.line(
            mon_df,
            x=mon_df.index,
            y=cols,
            title=f"{element} - {var} ({len(cols)} signal(s))",
            labels={"x": "Time (s)", "value": "Value", "variable": "Signal"},
        )
        _ = fig.update_layout(legend_title="Columns")
        _ = fig.show()

if unmatched:
    print("Unmatched columns:")
    for c in unmatched:
        print("  ", c)

# multiple runs for UQ

* 3-bus and hawaii cases: perturb H on multiple gen's
* perturb gen dispatch



## threebusbasic setup
* sampling H

In [ ]:
# === UQ config (ThreeBusBasic) ===
import importlib
import gridkit_utils

importlib.reload(gridkit_utils)
from gridkit_utils import generate_samples, make_run_dir, run_sample, collect_and_save

# --- case files ---
BASE_CASE_DIR = os.path.join(build_dir, "examples/PhasorDynamics/Tiny/ThreeBus/Basic/")

# --- output ---
UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/threebusbasic-v1"
SERIALIZE_MODE = "stacked"  # "stacked" (single results.parquet) or "per_run" (run_NNN.parquet per run)
UQ_OUT_PATH = (
    os.path.join(UQ_RUN_ROOT, "results.parquet")
    if SERIALIZE_MODE == "stacked"
    else os.path.join(UQ_RUN_ROOT, "runs")
)

# --- sampling ---
N_SAMPLES = 100
SEED = 42
SAMPLE_METHOD = "lhs"  # "lhs" or "random"

# --- solver overrides (applied to every run's .solver.json) ---
# None = use base case values unchanged
# tmax: simulation end time (s)
# events: list of {index, ...fields} — patch events by 0-based index
SOLVER_OVERRIDES = None
# SOLVER_OVERRIDES = {
#     "tmax": 20.0,
#     "events": [
#         {"index": 0, "time": 2.0},   # fault_on
#         {"index": 1, "time": 2.1},   # fault_off
#     ],
# }

# dist="uniform" with pct: lo = nominal*(1-pct), hi = nominal*(1+pct)
# dist="uniform" with lo/hi: fixed range
# dist="normal" with mean/std: Gaussian via LHS+ppf
PARAM_SPECS = [
    {
        "class": "Genrou",
        "id": "genrou_2_1",
        "param": "H",
        "dist": "uniform",
        "nominal": 2.7,
        "pct": 0.10,
    },
    {
        "class": "Genrou",
        "id": "genrou_3_1",
        "param": "H",
        "dist": "uniform",
        "nominal": 1.6,
        "pct": 0.10,
    },
]

MONITORS_BY_CLASS = {
    "bus": ["Vm", "Va"],
    "infinite_bus": ["Vm", "Va"],
    "genrou": ["delta", "omega"],
}

print(f"BASE_CASE_DIR:    {BASE_CASE_DIR}")
print(f"UQ_RUN_ROOT:      {UQ_RUN_ROOT}")
print(f"SERIALIZE_MODE:   {SERIALIZE_MODE}")
print(f"UQ_OUT_PATH:      {UQ_OUT_PATH}")
print(f"SOLVER_OVERRIDES: {SOLVER_OVERRIDES}")

## hawaii setup

In [ ]:
# === UQ config (Hawaii) ===
import importlib
import gridkit_utils

importlib.reload(gridkit_utils)
from gridkit_utils import generate_samples, make_run_dir, run_sample, collect_and_save

# --- case files (hawaii.json, not hawaii.case.json) ---
BASE_CASE_DIR = os.path.join(build_dir, "examples/PhasorDynamics/Medium/Hawaii/")

# --- output ---
UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v3"
SERIALIZE_MODE = "per_run"  # "stacked" (single results.parquet) or "per_run" (run_NNN.parquet per run)
UQ_OUT_PATH = (
    os.path.join(UQ_RUN_ROOT, "results.parquet")
    if SERIALIZE_MODE == "stacked"
    else os.path.join(UQ_RUN_ROOT, "runs")
)

# --- sampling ---
N_SAMPLES = 1000
SEED = 42
SAMPLE_METHOD = "lhs"  # "lhs" or "random"

# --- solver overrides ---
SOLVER_OVERRIDES = None
# SOLVER_OVERRIDES = {
#     "tmax": 20.0,
#     "events": [
#         {"index": 0, "time": 2.0},   # fault_on
#         {"index": 1, "time": 2.1},   # fault_off
#     ],
# }


# -----------------------------------------------------------------------
# H parameter specs: uncomment ONE block
# bus 2: H=3.69, bus 23: H=6.15, bus 34: H=4.35, bus 35: H=5.22
# -----------------------------------------------------------------------

# # --- option A: aleatoric -- uniform +/-10% of nominal ---
# _H_PCT = 0.10
# PARAM_SPECS = [
#     {
#         "class": "Genrou",
#         "id": "2_1",
#         "param": "H",
#         "dist": "uniform",
#         "nominal": 3.69,
#         "pct": _H_PCT,
#     },
#     {
#         "class": "Genrou",
#         "id": "23_1",
#         "param": "H",
#         "dist": "uniform",
#         "nominal": 6.15,
#         "pct": _H_PCT,
#     },
#     {
#         "class": "Genrou",
#         "id": "34_1",
#         "param": "H",
#         "dist": "uniform",
#         "nominal": 4.35,
#         "pct": _H_PCT,
#     },
#     {
#         "class": "Genrou",
#         "id": "35_1",
#         "param": "H",
#         "dist": "uniform",
#         "nominal": 5.22,
#         "pct": _H_PCT,
#     },
# ]

# --- option B: epistemic -- Gaussian std=12% of nominal ---
_H_STD_PCT = 0.12
PARAM_SPECS = [
    {
        "class": "Genrou",
        "id": "2_1",
        "param": "H",
        "dist": "normal",
        "mean": 3.69,
        "std": 3.69 * _H_STD_PCT,
    },
    {
        "class": "Genrou",
        "id": "23_1",
        "param": "H",
        "dist": "normal",
        "mean": 6.15,
        "std": 6.15 * _H_STD_PCT,
    },
    {
        "class": "Genrou",
        "id": "34_1",
        "param": "H",
        "dist": "normal",
        "mean": 4.35,
        "std": 4.35 * _H_STD_PCT,
    },
    {
        "class": "Genrou",
        "id": "35_1",
        "param": "H",
        "dist": "normal",
        "mean": 5.22,
        "std": 5.22 * _H_STD_PCT,
    },
]

# -----------------------------------------------------------------------

MONITORS_BY_CLASS = {
    "bus": ["Vm", "Va"],
    "genrou": ["delta", "omega"],
}

print(f"BASE_CASE_DIR:    {BASE_CASE_DIR}")
print(f"UQ_RUN_ROOT:      {UQ_RUN_ROOT}")
print(f"SERIALIZE_MODE:   {SERIALIZE_MODE}")
print(f"UQ_OUT_PATH:      {UQ_OUT_PATH}")
_dist = PARAM_SPECS[0]["dist"]
if _dist == "uniform":
    print(f"H dist: uniform +/-{PARAM_SPECS[0]['pct']*100:.0f}% of nominal")
    for s in PARAM_SPECS:
        print(
            f"  {s['id']}: nominal={s['nominal']:.2f}  [{s['nominal']*(1-s['pct']):.4f}, {s['nominal']*(1+s['pct']):.4f}]"
        )
else:
    print(
        f"H dist: normal, std={PARAM_SPECS[0]['std']/PARAM_SPECS[0]['mean']*100:.0f}% of nominal"
    )
    for s in PARAM_SPECS:
        print(f"  {s['id']}: mean={s['mean']:.2f}, std={s['std']:.4f}")

In [ ]:
# === Write experiment metadata to meta.yml ===
import yaml

os.makedirs(UQ_RUN_ROOT, exist_ok=True)

# build per-param sampling summary (human-readable ranges)
_dist = PARAM_SPECS[0]["dist"]
_sampling_summary = []
for s in PARAM_SPECS:
    if s["dist"] == "uniform":
        lo = s["nominal"] * (1 - s["pct"])
        hi = s["nominal"] * (1 + s["pct"])
        _sampling_summary.append(
            {
                "id": s["id"],
                "dist": "uniform",
                "nominal": s["nominal"],
                "pct": s["pct"],
                "lo": round(lo, 6),
                "hi": round(hi, 6),
            }
        )
    else:
        _sampling_summary.append(
            {
                "id": s["id"],
                "dist": "normal",
                "mean": s["mean"],
                "std": round(s["std"], 6),
                "std_pct": round(s["std"] / s["mean"], 4),
            }
        )

meta = {
    "case": os.path.basename(BASE_CASE_DIR.rstrip("/")),
    "base_case_dir": BASE_CASE_DIR,
    "run_root": UQ_RUN_ROOT,
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "sampling": {
        "n_samples": N_SAMPLES,
        "seed": SEED,
        "method": SAMPLE_METHOD,
        "dist_type": _dist,
        "params": _sampling_summary,
    },
    "serialize_mode": SERIALIZE_MODE,
    "solver_overrides": SOLVER_OVERRIDES,
    "monitors_by_class": MONITORS_BY_CLASS,
}
meta_path = os.path.join(UQ_RUN_ROOT, "meta.yml")
with open(meta_path, "w") as f:
    yaml.dump(meta, f, default_flow_style=False, sort_keys=False)
print(f"Wrote {meta_path}")
print(open(meta_path).read())

In [ ]:
# === Generate samples ===
os.makedirs(UQ_RUN_ROOT, exist_ok=True)
samples_df = generate_samples(PARAM_SPECS, N=N_SAMPLES, seed=SEED, method=SAMPLE_METHOD)
samples_df.to_csv(os.path.join(UQ_RUN_ROOT, "samples.csv"))
samples_df

In [ ]:
UQ_RUN_ROOT

In [ ]:
# # === ( 3-bus case only) Visualize LHS samples in H1–H2 parameter space ===
#######
# col_h1, col_h2 = samples_df.columns[0], samples_df.columns[1]
# s0, s1 = PARAM_SPECS[0], PARAM_SPECS[1]
# lo1, hi1 = s0["nominal"] * (1 - s0["pct"]), s0["nominal"] * (1 + s0["pct"])
# lo2, hi2 = s1["nominal"] * (1 - s1["pct"]), s1["nominal"] * (1 + s1["pct"])

# fig = px.scatter(
#     samples_df.reset_index(),
#     x=col_h1,
#     y=col_h2,
#     text="index",
#     title=f"LHS samples (N={N_SAMPLES}) in H parameter space",
#     labels={col_h1: f"H₁  ({s0['id']})", col_h2: f"H₂  ({s1['id']})"},
# )
# _ = fig.update_traces(textposition="top center", marker_size=8)
# _ = fig.add_shape(
#     type="rect",
#     x0=lo1,
#     x1=hi1,
#     y0=lo2,
#     y1=hi2,
#     line=dict(color="gray", dash="dash"),
#     fillcolor="rgba(0,0,0,0)",
# )
# _ = fig.update_layout(
#     xaxis=dict(range=[lo1 - 0.05 * (hi1 - lo1), hi1 + 0.05 * (hi1 - lo1)]),
#     yaxis=dict(range=[lo2 - 0.05 * (hi2 - lo2), hi2 + 0.05 * (hi2 - lo2)]),
# )
# _ = fig.show()

In [ ]:
# === Create run dirs + patch case.json ===
for i, row in samples_df.iterrows():
    d = make_run_dir(
        BASE_CASE_DIR,
        UQ_RUN_ROOT,
        i,
        row,
        PARAM_SPECS,
        MONITORS_BY_CLASS,
        solver_overrides=SOLVER_OVERRIDES,
    )
    print(f"  run_{i:03d}: {d}")

In [ ]:
# === Run all samples ===
failed = []
for i, row in samples_df.iterrows():
    run_dir = os.path.join(UQ_RUN_ROOT, f"run_{i:03d}")
    result = run_sample(run_dir, runner)
    status = "OK" if result.returncode == 0 else "FAILED"
    print(f"  run_{i:03d}: {status}")
    if result.returncode != 0:
        failed.append(i)
        print("    STDERR:", result.stderr[:300])

print(f"\nDone. {N_SAMPLES - len(failed)}/{N_SAMPLES} succeeded.")

In [ ]:
# === Collect results -> Parquet ===
#
# SERIALIZE_MODE options:
#   "stacked"  - all runs concatenated into one results.parquet
#                good for: small-medium N (<~500 runs), downstream analysis in a single df
#                          (pandas/pyarrow can filter by run_id without loading everything)
#                risk:     large N or many monitor signals -> multi-GB file, slow to load
#
#   "per_run"  - one run_NNN.parquet per run under UQ_RUN_ROOT/runs/
#                good for: large N (1000+), parallel post-processing, incremental collection
#                          (can re-collect after adding runs without rewriting everything)
#                risk:     slightly more overhead to open many files; need to concat manually
#                          for whole-ensemble analysis
#
# Override SERIALIZE_MODE here to produce a different format without changing the config cell.
SERIALIZE_MODE = "per_run"  # "stacked" or "per_run"
UQ_OUT_PATH = (
    os.path.join(UQ_RUN_ROOT, "results.parquet")
    if SERIALIZE_MODE == "stacked"
    else os.path.join(UQ_RUN_ROOT, "runs")
)

result = collect_and_save(UQ_RUN_ROOT, samples_df, UQ_OUT_PATH, mode=SERIALIZE_MODE)
if SERIALIZE_MODE == "stacked":
    results_df = result
    results_df
else:
    print(f"Written {len(result)} per-run files")
    for p in result[:5]:
        print(f"  {p}")
    if len(result) > 5:
        print(f"  ... ({len(result) - 5} more)")

In [ ]:
# === results size: in-memory and on-disk ===
if SERIALIZE_MODE == "stacked":
    mem_mb = results_df.memory_usage(deep=True).sum() / 1024**2
    disk_mb = os.path.getsize(UQ_OUT_PATH) / 1024**2
    print(f"results_df:  {results_df.shape[0]:,} rows × {results_df.shape[1]} cols")
    print(f"  in-memory: {mem_mb:.1f} MB")
    print(f"  parquet on disk: {disk_mb:.2f} MB")
else:
    import glob

    run_files = sorted(glob.glob(os.path.join(UQ_OUT_PATH, "run_*.parquet")))
    total_disk_mb = sum(os.path.getsize(f) for f in run_files) / 1024**2
    print(f"per_run: {len(run_files)} files in {UQ_OUT_PATH}")
    print(
        f"  total on disk: {total_disk_mb:.2f} MB  ({total_disk_mb/len(run_files):.2f} MB/run)"
    )

# plotting 
* for sanity check only
* more involved plotting is in gridkit_viv.ipynb

In [ ]:
# === Sanity check plot: 2 runs x 1 bus signal + 1 gen signal per var ===
PLOT_MAX_RUNS = 2
PLOT_MAX_BUS_N = 1
PLOT_MAX_GEN_N = 1

from collections import defaultdict

rng_plot = np.random.default_rng(SEED)

# Load a small subset of results depending on serialize mode
if SERIALIZE_MODE == "stacked":
    # already in memory from collect cell (has run_id column)
    _plot_df_full = results_df
else:
    # per_run: files have only time + signal cols; inject run_id from filename
    run_files = sorted(glob.glob(os.path.join(UQ_OUT_PATH, "run_*.parquet")))
    if not run_files:
        raise FileNotFoundError(f"No run_*.parquet files found in {UQ_OUT_PATH}")
    chosen_idx = sorted(
        rng_plot.choice(
            len(run_files), size=min(PLOT_MAX_RUNS, len(run_files)), replace=False
        )
    )
    frames = []
    for idx in chosen_idx:
        fpath = run_files[idx]
        run_id = int(
            os.path.basename(fpath).replace("run_", "").replace(".parquet", "")
        )
        df = pd.read_parquet(fpath)
        df.insert(0, "run_id", run_id)
        frames.append(df)
    _plot_df_full = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(chosen_idx)} run files: {[run_files[i] for i in chosen_idx]}")

param_cols = list(samples_df.columns)
skip_cols = {"run_id", "time", "Solver Status"} | set(param_cols)
mon_cols = [c for c in _plot_df_full.columns if c not in skip_cols]

all_run_ids = sorted(_plot_df_full["run_id"].unique())
plot_run_ids = list(
    rng_plot.choice(
        all_run_ids, size=min(PLOT_MAX_RUNS, len(all_run_ids)), replace=False
    )
)
print(f"Runs: {plot_run_ids}")
plot_df = _plot_df_full[_plot_df_full["run_id"].isin(plot_run_ids)]

col_groups = defaultdict(list)
for col in mon_cols:
    parts = col.split("_")
    key = ("Bus", parts[-1]) if parts[0] == "Bus" else (parts[0], parts[-1])
    col_groups[key].append(col)

for (elem, var), cols in sorted(col_groups.items()):
    max_n = PLOT_MAX_BUS_N if elem == "Bus" else PLOT_MAX_GEN_N
    plot_cols = list(rng_plot.choice(cols, size=min(max_n, len(cols)), replace=False))

    melted = plot_df[["time", "run_id"] + plot_cols].melt(
        id_vars=["time", "run_id"], var_name="signal", value_name=var
    )
    fig = px.line(
        melted,
        x="time",
        y=var,
        color="signal",
        line_group="run_id",
        title=f"{elem} {var} - {plot_cols}, {len(plot_run_ids)} runs",
        labels={"time": "Time (s)", var: var},
    )
    _ = fig.update_traces(opacity=0.7)
    _ = fig.update_layout(legend_title="signal")
    _ = fig.show()

In [ ]:
# # === Query parquet directly — run 3, Bus 2 Va ===
# import pyarrow.parquet as pq
# import pyarrow.compute as pc

# tbl = pq.read_table(UQ_OUT_PARQUET, filters=[("run_id", "=", 3)])
# va_cols = [c for c in tbl.schema.names if c.startswith("Bus_") and c.endswith("_Va")]
# run3_bus2_va = tbl.select(["time"] + va_cols).to_pandas()
# run3_bus2_va

# # === Query parquet — all runs, all Genrou omega signals ===
# tbl = pq.read_table(UQ_OUT_PARQUET)
# omega_cols = [c for c in tbl.schema.names if c.endswith("_omega")]
# all_omega = tbl.select(["run_id", "time"] + omega_cols).to_pandas()
# all_omega

# end